# Walk-Forward Research Pipeline

This notebook adds a separate research layer on top of the frozen UFC modeling repo. It does **not** modify the locked final hold-out benchmark outputs. Instead, it uses the earlier pre-test, odds-matched data to run an expanding walk-forward study with calibration and market benchmark comparisons.

This research layer also uses the audited post-2009 modeling universe only, meaning the predictive sample starts at `2010-01-01`. The goal is to study calibration and market benchmarking without reintroducing the pre-2010 target-definition break.




## Why this notebook exists

A single train/validation/test split is useful, but it can still leave us exposed to time-period luck. This notebook asks a more quant-style question: if we had rolled the model forward through time, would the raw probabilities, calibrated probabilities, or market-implied probabilities have looked strongest out of sample?


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import json

import matplotlib.pyplot as plt
import pandas as pd

from src.train_baseline_models import run_walk_forward_market_research

outputs_dir = repo_root / "outputs"
figures_dir = outputs_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
plt.style.use("default")


## Run the research workflow

The workflow uses the frozen final feature set and the validation-selected Random Forest configuration. It then builds annual walk-forward folds inside the pre-test period only: expanding train, one-year calibration, next-year test.


In [ ]:
results = run_walk_forward_market_research()
results["comparison"]


## Saved summary

The text summary captures the main takeaways from the walk-forward comparison.


In [ ]:
print((outputs_dir / "walk_forward_market_benchmark_summary.txt").read_text())


## Fold design

Each row below is one out-of-sample annual fold. The model is trained on all earlier years, calibrated on the immediately previous year, and then evaluated on the next year.

The walk-forward folds therefore start only after the modeling cutoff and only on odds-matched fights inside that audited era.



In [ ]:
fold_summary = pd.read_csv(outputs_dir / "walk_forward_fold_summary.csv")
fold_summary


## Aggregate method comparison

These metrics are computed on the concatenated out-of-sample predictions from all walk-forward folds. Lower log loss and Brier score are better; higher ROC AUC is better.


In [ ]:
comparison = pd.read_csv(outputs_dir / "walk_forward_model_comparison.csv")
comparison


### Probability metric comparison

This chart compares the raw market benchmark, the raw Random Forest probabilities, calibrated versions of the model, and model-market blends.


In [ ]:
plot_df = comparison.set_index("model_name").loc[[
    "market_novig",
    "rf_raw",
    "rf_sigmoid",
    "rf_isotonic",
    "rf_sigmoid_market_blend",
    "rf_isotonic_market_blend",
]].reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(plot_df["model_name"], plot_df["log_loss"], color="tab:blue")
axes[0].set_title("Walk-Forward Log Loss")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(plot_df["model_name"], plot_df["brier_score"], color="tab:green")
axes[1].set_title("Walk-Forward Brier Score")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(plot_df["model_name"], plot_df["roc_auc"], color="tab:orange")
axes[2].set_title("Walk-Forward ROC AUC")
axes[2].tick_params(axis="x", rotation=45)

for ax in axes:
    ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
fig.savefig(figures_dir / "07_walk_forward_method_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Year-by-year probability performance

This view helps us see whether any method only looked good because of one especially favorable year.


In [ ]:
yearly = pd.read_csv(outputs_dir / "walk_forward_yearly_metrics.csv")
focus_methods = ["market_novig", "rf_raw", "rf_sigmoid", "rf_isotonic_market_blend"]
yearly_plot = yearly[yearly["model_name"].isin(focus_methods)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for method_name in focus_methods:
    method_df = yearly_plot[yearly_plot["model_name"] == method_name].sort_values("test_year")
    ax.plot(method_df["test_year"], method_df["log_loss"], marker="o", linewidth=2, label=method_name)

ax.set_title("Walk-Forward Log Loss by Test Year")
ax.set_xlabel("Test Year")
ax.set_ylabel("Log Loss")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / "07_walk_forward_yearly_log_loss.png", dpi=150, bbox_inches="tight")
plt.show()


## Calibration comparison

The point here is not just ranking fights correctly. We also want the probability scale itself to be credible. These decile curves compare the realized win rate against the mean predicted probability for several methods.


In [ ]:
calibration = pd.read_csv(outputs_dir / "walk_forward_calibration_comparison.csv")
calibration_methods = ["market_novig", "rf_raw", "rf_sigmoid", "rf_isotonic_market_blend"]

fig, ax = plt.subplots(figsize=(7, 6))
for method_name in calibration_methods:
    method_df = calibration[calibration["model_name"] == method_name]
    ax.plot(
        method_df["mean_predicted_probability"],
        method_df["observed_red_win_rate"],
        marker="o",
        linewidth=2,
        label=method_name,
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1, label="perfect calibration")
ax.set_title("Walk-Forward Calibration Comparison")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Red Win Rate")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / "07_walk_forward_calibration_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Blend weights

Blend weights are chosen on the fold calibration year only. A higher weight means the workflow leaned more on the model; a lower weight means it leaned more on the market.


In [ ]:
blend_weights = pd.read_csv(outputs_dir / "walk_forward_blend_weights.csv")
blend_weights


## Interpretation

A few things matter more here than raw classification accuracy:

- whether calibration helps the Random Forest probability scale
- whether the market still dominates on pure probability quality
- whether a blend beats either side alone
- whether the results stay stable across years instead of depending on one lucky regime

This is still research, not a new frozen production benchmark. The final consumed hold-out test remains a separate layer.


## Why this matters for the betting layer

This notebook sits between the modeling notebooks and the betting notebook. If the market still dominates on walk-forward probability metrics, then a negative betting backtest becomes much less surprising. In other words, this workflow helps us separate two different questions:

- does the model contain some predictive signal?
- does the model add enough information beyond the market to create tradable probability edge?

So far, the answer looks closer to "yes" for the first question than for the second.
